In [ ]:
!pip install torch esm pandas
!pip install fair-esm --upgrade
!pip install biopython

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import esm
import pandas as pd
from Bio import SeqIO

# Load ESM-2 320-dimensional model (6 layers, 8M params)
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()  # Set to eval mode (no gradients)

# Load sequences from FASTA
fasta_file = "/content/drive/MyDrive/Colab Notebooks/AntiCancer-Training.fasta"
sequences = [(record.id, str(record.seq)) for record in SeqIO.parse(fasta_file, "fasta")]

# Prepare data
batch_labels, batch_strs, batch_tokens = batch_converter(sequences)

# Forward pass to extract embeddings
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[6])  # Layer 6 is the final layer in this model
    token_representations = results["representations"][6]

# Average token embeddings (excluding special tokens)
sequence_representations = []
for i, (_, seq) in enumerate(sequences):
    representation = token_representations[i, 1:len(seq)+1].mean(0)
    sequence_representations.append(representation)

# Convert to numpy array
sequence_representations = torch.stack(sequence_representations).numpy()

# Save to DataFrame
df = pd.DataFrame(sequence_representations, index=[s[0] for s in sequences])
df.to_csv("ESM320_features.csv")

print("✅ ESM-320 features extracted and saved!")
